# Chapter 14 - Model, Design, and Base Lattices

This chapter translates the original Tao workflow for **design**, **model**, and **base** lattices into a SciBmad/Julia workflow.

The original chapter is mostly about bookkeeping: Tao keeps several lattice states at the same time so that the user can change one lattice and compare it to another. SciBmad does not provide the Tao command interface, but Julia makes the same idea explicit with ordinary objects and copied snapshots.

In this notebook we will:

1. define the three lattice roles and compare their Bmad/Tao and SciBmad meanings;
2. reproduce the small Chapter 14 example lattice in SciBmad;
3. change model-lattice parameters and plot `model - design`;
4. save a base lattice and plot `model - base`.

In [ ]:
using SciBmad
using CairoMakie
using LinearAlgebra
using Printf

## Three Lattice States

| Concept | Bmad/Tao meaning | SciBmad representation in this notebook |
|---|---|---|
| Design lattice | The lattice read from the input file. Tao keeps its parameters fixed. | `design`, a lattice object created from the original settings and then left unchanged. |
| Model lattice | The working lattice. Tao commands such as `set element` and `change element` modify this lattice. | `model`, a `deepcopy(design)` that we intentionally mutate during calculations. |
| Base lattice | A user-selected reference copy, often set equal to the current model lattice. | `base`, another copied snapshot, updated with `base = deepcopy(model)`. |

This is one of the places where SciBmad is more explicit than Tao. Tao creates and manages these states inside the program. In SciBmad, the notebook owns the state, so the exact moment when a reference is saved is visible in the Julia code.

The original Chapter 14 example uses the simple lattice:

```bmad
beginning[beta_a] = 10.
beginning[beta_b] = 10.
beginning[e_tot] = 10e6

parameter[geometry] = open
parameter[particle] = electron

d: drift, L = 0.5
b: sbend, L = 0.5, g = 1, e1 = 0.1, dg = 0.001
q: quadrupole, L = 0.6, k1 = 0.23

lat: line = (d, b, q)
use, lat
```

SciBmad can represent the drift, bend, and quadrupole directly. The Bmad `dg = 0.001` bend-field error is represented by setting the bend reference curvature to `g_ref = 1.0` and the actual normal dipole strength to `Kn0 = 1.001`. Tao's `hkick` and `vkick` attributes are represented here by zero-length thin-kick markers at the entrance of the corresponding element. This keeps the examples simple while preserving the intended behavior: the model lattice receives local horizontal or vertical angular kicks.

In [ ]:
const CH13_SPECIES = Species("electron")
const CH13_E_REF = 10e6

const CH13_N_DRIFT = 10
const CH13_N_BEND = 40
const CH13_N_QUAD = 40

function thin_kick_map(v, q::Nothing, p)
    hkick, vkick = p
    return ((v[1], v[2] + hkick, v[3], v[4] + vkick, v[5], v[6]), q)
end

function thin_kick(name; hkick=0.0, vkick=0.0)
    marker = Marker(name=name)
    marker.transport_map = thin_kick_map
    marker.transport_map_params = (hkick, vkick)
    return marker
end

function chapter14_drift_slices()
    return [Drift(name=@sprintf("D_%02d", i), L=0.5 / CH13_N_DRIFT) for i in 1:CH13_N_DRIFT]
end

function chapter14_bend_slices()
    return [
        SBend(
            name=@sprintf("B_%02d", i),
            L=0.5 / CH13_N_BEND,
            e1=(i == 1 ? 0.1 : 0.0),
            g_ref=1.0,
            Kn0=1.001,
        )
        for i in 1:CH13_N_BEND
    ]
end

function chapter14_quad_slices()
    return [Quadrupole(name=@sprintf("Q_%02d", i), L=0.6 / CH13_N_QUAD, Kn1=0.23) for i in 1:CH13_N_QUAD]
end

function build_chapter14_lattice(; b_vkick=0.0, q_hkick=0.0, q_vkick=0.0)
    b_kick = thin_kick("B_KICK"; vkick=b_vkick)
    q_kick = thin_kick("Q_KICK"; hkick=q_hkick, vkick=q_vkick)

    return Beamline(
        vcat(chapter14_drift_slices(), [b_kick], chapter14_bend_slices(), [q_kick], chapter14_quad_slices());
        species_ref=CH13_SPECIES,
        E_ref=CH13_E_REF,
    )
end

mutable struct LatticeState
    design
    model
    base
end

function chapter14_state()
    design = build_chapter14_lattice()
    return LatticeState(design, deepcopy(design), deepcopy(design))
end

In [ ]:
function element_by_name(beamline, name)
    for element in beamline.line
        string(element.name) == name && return element
    end
    error("Element named $(name) was not found.")
end

function set_kick!(beamline, name; hkick=nothing, vkick=nothing)
    element = element_by_name(beamline, name)
    old_hkick, old_vkick = element.transport_map_params
    new_hkick = isnothing(hkick) ? old_hkick : hkick
    new_vkick = isnothing(vkick) ? old_vkick : vkick
    element.transport_map_params = (new_hkick, new_vkick)
    return beamline
end

function change_kick!(beamline, name; dhkick=0.0, dvkick=0.0)
    element = element_by_name(beamline, name)
    old_hkick, old_vkick = element.transport_map_params
    element.transport_map_params = (old_hkick + dhkick, old_vkick + dvkick)
    return beamline
end

function reference_bunch(beamline, coordinates)
    coordinates = reshape(coordinates, 1, 6)

    if hasproperty(beamline, :p_over_q_ref)
        return Bunch(coordinates; species=beamline.species_ref, p_over_q_ref=beamline.p_over_q_ref)
    elseif hasproperty(beamline, :R_ref)
        return Bunch(coordinates; species=beamline.species_ref, R_ref=beamline.R_ref)
    else
        return Bunch(coordinates; species=beamline.species_ref)
    end
end

function track_orbit(beamline; v0=zeros(6))
    bunch = reference_bunch(beamline, copy(v0))
    history = zeros(length(beamline.line) + 1, 6)
    history[1, :] .= bunch.coords.v[1, :]

    s = zeros(length(beamline.line) + 1)
    for (i, element) in enumerate(beamline.line)
        track!(bunch, element)
        history[i + 1, :] .= bunch.coords.v[1, :]
        s[i + 1] = s[i] + element.L
    end

    return (; s, x=history[:, 1], px=history[:, 2], y=history[:, 3], py=history[:, 4], history)
end

In [ ]:
function readable_ylims!(ax, orbit; minimum_span_mm=0.08)
    values = 1e3 .* vcat(orbit.x, orbit.y)
    lo, hi = extrema(values)
    center = 0.5 * (lo + hi)
    span = max(hi - lo, minimum_span_mm)
    ylims!(ax, center - 0.55 * span, center + 0.55 * span)
end

function add_orbit_panel!(fig, row, title, orbit; color_x=:royalblue3, color_y=:darkorange2)
    ax = Axis(
        fig[row, 1],
        xlabel="s [m]",
        ylabel="orbit [mm]",
        title=title,
    )
    lines!(ax, orbit.s, 1e3 .* orbit.x; color=color_x, linewidth=2, label="x")
    scatter!(ax, orbit.s, 1e3 .* orbit.x; color=color_x, markersize=8)
    lines!(ax, orbit.s, 1e3 .* orbit.y; color=color_y, linewidth=2, label="y")
    scatter!(ax, orbit.s, 1e3 .* orbit.y; color=color_y, markersize=8)
    xlims!(ax, -0.02, orbit.s[end] + 0.02)
    readable_ylims!(ax, orbit)
    axislegend(ax; position=:lt)
    return ax
end

function orbit_difference(a, b)
    return (; s=a.s, x=a.x .- b.x, px=a.px .- b.px, y=a.y .- b.y, py=a.py .- b.py)
end

function print_orbit_difference_summary(state; reference=:design)
    model_orbit = track_orbit(state.model)
    reference_orbit = reference == :base ? track_orbit(state.base) : track_orbit(state.design)
    difference = orbit_difference(model_orbit, reference_orbit)
    label = reference == :base ? "model - base" : "model - design"

    @printf(
        "%s at lattice end: dx = %+8.4f mm, dy = %+8.4f mm, dpx = %+9.3e, dpy = %+9.3e\n",
        label,
        1e3 * difference.x[end],
        1e3 * difference.y[end],
        difference.px[end],
        difference.py[end],
    )
end

function plot_model_design_base(state; bottom=:design)
    design_orbit = track_orbit(state.design)
    model_orbit = track_orbit(state.model)
    base_orbit = track_orbit(state.base)

    bottom_orbit = bottom == :base ? orbit_difference(model_orbit, base_orbit) : design_orbit
    bottom_title = bottom == :base ? "Orbit [model - base]" : "Orbit [design]"

    fig = Figure(size=(760, 780))
    ax1 = add_orbit_panel!(fig, 1, "Orbit [model - design]", orbit_difference(model_orbit, design_orbit))
    ax2 = add_orbit_panel!(fig, 2, "Orbit [model]", model_orbit)
    ax3 = add_orbit_panel!(fig, 3, bottom_title, bottom_orbit)
    linkxaxes!(ax1, ax2, ax3)
    return fig
end

At startup, Tao's model lattice is equal to the design lattice. The SciBmad equivalent is to make `model` as a fresh copy of `design`. Therefore `model - design` should initially be zero.

In [ ]:
state = chapter14_state()
fig = plot_model_design_base(state)
print_orbit_difference_summary(state; reference=:design)
fig

## 14.1 Changing Model Parameters

In Tao, ordinary lattice-parameter edits apply to the model lattice. The Chapter 14 commands are:

```text
Tao> change element b vkick -0.0005
Tao> set element q hkick = 0.001
```

Here `change` means add a delta to the current value, while `set` means replace the current value. In this notebook, the corresponding SciBmad operations are explicit Julia functions acting on `state.model`.

In [ ]:
# Tao: change element b vkick -0.0005
change_kick!(state.model, "B_KICK"; dvkick=-0.0005)

# Tao: set element q hkick = 0.001
set_kick!(state.model, "Q_KICK"; hkick=0.001)

fig = plot_model_design_base(state)
print_orbit_difference_summary(state; reference=:design)
fig

The design lattice has not changed. Only the model lattice has changed, so the top plot now shows a nonzero `model - design` orbit difference. This is the same conceptual role as Tao's `set plot ... component = model - design` command.

## 14.2 Using the Base Lattice

The base lattice is useful when the desired reference is no longer the original design lattice. In Tao, the command

```text
Tao> set lattice base = model
```

copies the current model into the base lattice. The SciBmad equivalent is simply:

```julia
state.base = deepcopy(state.model)
```

After saving that reference, further changes to `state.model` can be viewed against either `design` or `base`.

In [ ]:
# Tao: set lattice base = model
state.base = deepcopy(state.model)

# Tao: set ele q vkick = 5e-4
set_kick!(state.model, "Q_KICK"; vkick=5e-4)

fig = plot_model_design_base(state; bottom=:base)
print_orbit_difference_summary(state; reference=:design)
print_orbit_difference_summary(state; reference=:base)
fig

The top panel still compares the current model to the original design lattice. The bottom panel compares the current model to the saved base snapshot. These two differences answer different questions:

- `model - design` shows all accumulated changes since the input lattice was created.
- `model - base` shows only the changes made after the base snapshot was saved.

That is the practical purpose of the base lattice in Tao, and the same idea carries over cleanly to SciBmad as an explicit copied reference state.

## 14.3 Exercises

1. **Orbit plots.** Modify the middle graph in Figure 27 so that the vertical-orbit curve becomes the horizontal orbit of the design lattice. In the SciBmad version, this means building one panel from explicitly selected orbit arrays rather than using Tao plot-component commands.

2. **Alias command.** The original Tao exercise defines an alias named `setit` so that typing `setit 1e-6` is equivalent to `set ele q vkick = 1e-6`. In SciBmad, write the equivalent small Julia function that sets the model lattice `Q_KICK` vertical kick.

3. **Adjust the tune using `set tune`.** The Tao command `set tune -mask *,~QF,~QD 0.08 0.14` scales selected focusing and defocusing quadrupoles to reach target tunes. In SciBmad, implement the same idea with explicit QF/QD family knobs, match the target tunes, and plot the beta beating between the tuned model lattice and the original design lattice.

Reference solutions are provided in `lattices/chapter_14/`.
